# 04 — Evaluación del clasificador SVM

Análisis a fondo del modelo entrenado por `scripts/compare_svm_rf.py`. Carga el SVM con grid search (las 8 categorías del proyecto) e incluye:

1. Métricas globales y por clase.
2. **Matriz de confusión normalizada** (mejor que la cruda para datasets desbalanceados).
3. Predicción sobre regiones reales del pipeline.
4. **Galería de errores** (qué imágenes falla el SVM y por qué).
5. **t-SNE del espacio de features** para ver visualmente cómo se separan las 8 categorías.

**Pre-requisitos:**
```bash
python scripts/build_dataset.py
python scripts/compare_svm_rf.py
```

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import joblib
import matplotlib.pyplot as plt
import cv2
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

from src.features import extract_features
from src.preprocessing import preprocess
from src.region_proposal import propose_regions
from src.utils.io_utils import load_image
from src.classification import split_dataset

plt.rcParams['figure.dpi'] = 90

## 1. Cargar modelo y datos

In [ ]:
MODEL_PATH = Path('../results/models/svm_categoria_gridsearch.joblib')
DATASET_DIR = Path('../results/dataset')

if not MODEL_PATH.is_file():
    print(f'❌  No existe {MODEL_PATH}')
    print('Ejecuta primero: python scripts/compare_svm_rf.py')
else:
    bundle = joblib.load(MODEL_PATH)
    model = bundle['model']
    print(f'✓  Modelo cargado.')
    print(f'   Mejor combinación de hiperparámetros: {bundle.get("best_params", "N/A")}')
    metrics = bundle.get('metrics', {})
    if metrics:
        print(f'   Métricas guardadas:')
        for k, v in metrics.items():
            print(f'     {k:20s} {v:.4f}')

## 2. Cargar predicciones de test guardadas

In [ ]:
y_test = np.load(DATASET_DIR / 'y_test.npy', allow_pickle=True)
y_pred = np.load(DATASET_DIR / 'y_pred.npy', allow_pickle=True)

print(f'Tamaño del test: {len(y_test)} imágenes')
print(f'Aciertos: {(y_test == y_pred).sum()}  ({100*(y_test == y_pred).mean():.1f}%)')
print(f'Fallos: {(y_test != y_pred).sum()}')

## 3. Matriz de confusión cruda y normalizada

La normalizada es más legible cuando las clases están desbalanceadas: cada fila suma 1 y muestra qué porcentaje de la clase real va a cada predicción.

In [ ]:
labels = sorted(set(y_test))
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Matriz cruda
ax = axes[0]
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right'); ax.set_yticklabels(labels)
ax.set_xlabel('Predicción'); ax.set_ylabel('Real')
ax.set_title('Matriz de confusión (cruda)', fontsize=12)
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=9,
                color='white' if cm[i, j] > cm.max()/2 else 'black')
plt.colorbar(im, ax=ax, fraction=0.046)

# Matriz normalizada
ax = axes[1]
im = ax.imshow(cm_norm, cmap='Oranges', vmin=0, vmax=1)
ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right'); ax.set_yticklabels(labels)
ax.set_xlabel('Predicción'); ax.set_ylabel('Real')
ax.set_title('Matriz de confusión (normalizada por fila)', fontsize=12)
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f'{cm_norm[i, j]:.2f}', ha='center', va='center', fontsize=9,
                color='white' if cm_norm[i, j] > 0.5 else 'black')
plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout(); plt.show()
print()
print('Classification report:')
print(classification_report(y_test, y_pred, zero_division=0))

## 4. Predicción sobre regiones reales del pipeline

Cogemos una imagen completa, le aplicamos region proposal y para cada caja predecimos la categoría. Es una vista previa del pipeline integrado en F6.

In [ ]:
DATA_ROOT = Path('../data/external/combined')
test_class = 'Apple'  # cambia por una clase del dataset combinado

cat_dir = DATA_ROOT / test_class
imgs = sorted(cat_dir.glob('*.jpg')) + sorted(cat_dir.glob('*.png'))
if not imgs:
    print(f'No hay imágenes en {test_class}, prueba con otra')
else:
    img = load_image(imgs[0])
    img_pre = preprocess(img)
    proposals = propose_regions(img_pre)

    # Predecir cada caja
    predicted = []
    for p in proposals:
        crop = img_pre[p.y:p.y+p.h, p.x:p.x+p.w]
        if crop.size == 0:
            continue
        features = extract_features(crop)
        pred = model.predict(features.reshape(1, -1))[0]
        proba = float(model.predict_proba(features.reshape(1, -1))[0].max())
        predicted.append((p, pred, proba))

    # Paleta para las 8 categorías del proyecto
    palette = {
        'fruta':   (50, 200, 50),
        'verdura': (255, 200, 0),
        'brick':   (0, 100, 255),
        'lata':    (200, 0, 200),
        'botella': (0, 200, 200),
        'caja':    (180, 100, 50),
        'bolsa':   (200, 200, 200),
        'tarro':   (50, 50, 220),
    }
    bgr = cv2.cvtColor(img_pre, cv2.COLOR_RGB2BGR)
    for p, pred, proba in predicted:
        color = palette.get(pred, (255, 255, 255))
        cv2.rectangle(bgr, (p.x, p.y), (p.x+p.w, p.y+p.h), color, 2)
        label = f'{pred} {proba:.2f}'
        cv2.putText(bgr, label, (p.x+2, p.y-4), cv2.FONT_HERSHEY_SIMPLEX,
                    0.5, color, 1, cv2.LINE_AA)
    annotated = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(img_pre); axes[0].set_title('Preprocesada'); axes[0].axis('off')
    axes[1].imshow(annotated)
    axes[1].set_title(f'{len(predicted)} cajas con predicción SVM')
    axes[1].axis('off')
    plt.tight_layout(); plt.show()

    print('\nTop-5 predicciones por confianza:')
    for p, pred, proba in sorted(predicted, key=lambda x: -x[2])[:5]:
        print(f'  {pred:10s} ({proba:.2%}) en ({p.x},{p.y},{p.w},{p.h})')

## 5. Galería de errores

Cargamos las imágenes mal clasificadas y vemos qué confunde al SVM. Esto da pistas para mejorar el modelo en F6.

Truco: reproducimos exactamente el mismo split del entrenamiento (mismo `random_state=42`) para recuperar las paths de las imágenes de test.

In [ ]:
# Reproducir el split exacto del entrenamiento
X = np.load(DATASET_DIR / 'X.npy')
y = np.load(DATASET_DIR / 'y.npy', allow_pickle=True)
paths = np.load(DATASET_DIR / 'paths.npy', allow_pickle=True)

# split_dataset usa internamente train_test_split con random_state=42 en dos
# pasos: primero separa test (15%) y luego val. Para recuperar las paths del
# test reproducimos el mismo paso 1.
indices = np.arange(len(y))
_, idx_test = train_test_split(indices, test_size=0.15, stratify=y, random_state=42)

# Comprobar que coincide con y_test guardado
y_test_check = y[idx_test]
assert (y_test_check == y_test).all(), 'El split no coincide. ¿Cambió random_state?'
print(f'✓  Split reproducido. {len(idx_test)} imágenes de test.')

In [ ]:
wrong_mask = y_pred != y_test
wrong_idx_in_test = np.where(wrong_mask)[0]
print(f'Total errores: {len(wrong_idx_in_test)} / {len(y_test)} ({100*wrong_mask.mean():.1f}%)')

# Información detallada por par (real → predicha)
from collections import Counter
error_pairs = Counter()
for i in wrong_idx_in_test:
    error_pairs[(y_test[i], y_pred[i])] += 1

print('\nTop 10 pares de confusión (real → predicha):')
for (real, pred), n in error_pairs.most_common(10):
    print(f'  {real:10s} → {pred:10s}  {n:3d} veces')

In [ ]:
# Mostrar 12 errores aleatorios con la imagen, etiqueta real y predicha
import random
random.seed(7)

n_show = 12
sample_wrong = random.sample(list(wrong_idx_in_test), min(n_show, len(wrong_idx_in_test)))

cols = 4
rows = (n_show + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(3.2 * cols, 3.2 * rows))
axes = axes.flatten()

for ax, i in zip(axes, sample_wrong):
    real = y_test[i]
    pred = y_pred[i]
    path = paths[idx_test[i]]
    try:
        img = load_image(Path(path))
        ax.imshow(img)
    except Exception as e:
        ax.text(0.5, 0.5, f'Error\n{e}', ha='center', va='center')
    ax.set_title(f'real: {real}\npred: {pred}', fontsize=9, color='#C62828')
    ax.axis('off')

# Apagar ejes sobrantes
for ax in axes[len(sample_wrong):]:
    ax.axis('off')

plt.suptitle(f'Galería de errores ({len(sample_wrong)} de {len(wrong_idx_in_test)})',
             fontsize=12, y=1.0)
plt.tight_layout(); plt.show()

## 6. t-SNE del espacio de features

Reducimos los 1806 dimensiones del espacio de features a 2D y los pintamos coloreados por categoría. Si las clases se separan visualmente, es prueba gráfica de que las features funcionan.

t-SNE puede tardar ~1-2 minutos sobre todo el dataset; aquí muestreamos 1500 puntos para que sea rápido y legible.

In [ ]:
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

# Subsample para que t-SNE no tarde una eternidad
sample_size = min(1500, len(y))
sample_idx = np.random.RandomState(42).choice(len(y), sample_size, replace=False)
X_sub = X[sample_idx]
y_sub = y[sample_idx]

# Escalar como en el pipeline
scaler = StandardScaler().fit(X)
X_scaled = scaler.transform(X_sub)

print(f'Calculando t-SNE sobre {sample_size} muestras (~30-60 s)...')
tsne = TSNE(n_components=2, perplexity=30, random_state=42,
            init='pca', learning_rate='auto', n_iter=1000)
X_2d = tsne.fit_transform(X_scaled)
print('✓  t-SNE listo')

In [ ]:
# Pintar
fig, ax = plt.subplots(figsize=(11, 8))
categories_unique = sorted(set(y_sub))
# Paleta: usar los mismos colores del proyecto, pero como tuplas RGB normalizadas para matplotlib
palette_mpl = {
    'fruta':   (0.20, 0.78, 0.20),
    'verdura': (1.00, 0.78, 0.00),
    'brick':   (1.00, 0.39, 0.00),
    'lata':    (0.78, 0.00, 0.78),
    'botella': (0.00, 0.78, 0.78),
    'caja':    (0.20, 0.39, 0.71),
    'bolsa':   (0.50, 0.50, 0.50),
    'tarro':   (0.86, 0.20, 0.20),
}

for cat in categories_unique:
    mask = y_sub == cat
    color = palette_mpl.get(cat, (0.5, 0.5, 0.5))
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], c=[color], label=f'{cat} ({mask.sum()})',
               s=20, alpha=0.65, edgecolors='black', linewidths=0.3)

ax.set_xlabel('Dimensión 1 (t-SNE)')
ax.set_ylabel('Dimensión 2 (t-SNE)')
ax.set_title(f't-SNE del espacio de features (1806 dims → 2D, {sample_size} muestras)',
             fontsize=12)
ax.legend(title='Categoría', loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Conclusiones

Lo que esperamos confirmar al ejecutar:

- **La matriz normalizada** muestra los pares de confusión más relevantes (ej: bolsa ↔ caja, lata ↔ botella). Coincide con lo que ya sabemos del análisis de errores.
- **La galería de errores** ilustra por qué falla: productos visualmente parecidos (bolsa con etiqueta colorida confundida con caja, lata reflectante confundida con botella).
- **El t-SNE** debería mostrar nubes parcialmente separadas; las clases con mejor F1 (fruta, brick) forman clusters claros, y las que peor (lata, bolsa, botella) están más mezcladas. Esto es **prueba gráfica** del análisis numérico.

**Próximo paso:** F4 — capturar el dataset Monster propio para entrenar la rama Deep Learning.